# Titanic Feature Engineering Notebook

This notebook documents data cleaning, feature engineering, and feature selection decisions for Assignment 2.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from pathlib import Path

root = Path.cwd().parents[0] if Path.cwd().name == 'notebooks' else Path.cwd()
data_dir = root / 'data'
train_path = data_dir / 'train.csv'
train = pd.read_csv(train_path)
train.head()

## Part 1: Data Cleaning Decisions

- Check missingness and decide imputation strategy
- Handle outliers in `Age` and `Fare`
- Standardize categorical consistency and remove duplicates

In [ ]:
train.isna().sum().sort_values(ascending=False)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.boxplot(y=train['Age'], ax=axes[0])
axes[0].set_title('Age Outliers')
sns.boxplot(y=train['Fare'], ax=axes[1])
axes[1].set_title('Fare Outliers')
plt.tight_layout()

## Part 2: Feature Engineering

Create required features: `FamilySize`, `IsAlone`, `Title`, `Deck`, `AgeGroup`, and `FarePerPerson`.

In [ ]:
from scripts.data_cleaning import clean_titanic_data
from scripts.feature_engineering import build_features

cleaned, cleaning_decisions = clean_titanic_data(train)
engineered, scaler = build_features(cleaned)

print(cleaning_decisions)
print('Cleaned shape:', cleaned.shape)
print('Engineered shape:', engineered.shape)

In [ ]:
engineered.head()

## Part 3: Feature Selection

Correlation filtering + feature importance ranking + optional RFE.

In [ ]:
from scripts.feature_selection import drop_high_correlation, feature_importance_ranking, run_rfe

y = engineered['Survived']
x = engineered.drop(columns=['Survived']).select_dtypes(include=[np.number])
x_uncorr, dropped = drop_high_correlation(x, threshold=0.9)
ranking = feature_importance_ranking(x_uncorr, y)
rfe_cols = run_rfe(x_uncorr, y, n_features_to_select=12)

print('Dropped by correlation:', dropped)
display(ranking.head(15))
print('RFE features:', rfe_cols)